# Evaluate MNIST Models

In [ ]:
%pip install --quiet mlflow==3.6.0 tensorflow==2.20.0 xgboost==3.1.3 scikit-learn==1.8.0
dbutils.library.restartPython()

In [ ]:
import numpy as np
import os
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.tensorflow
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.set_registry_uri('databricks-uc')

# Get volume_path from DAB parameter (or use default)
try:
    volume_path = dbutils.widgets.get('volume_path')
except:
    volume_path = '/Volumes/ai_ml_in_practice/mnist_data/processed'

print(f'Volume path: {volume_path}\n')

# Load test data
x_test_flat = np.load(os.path.join(volume_path, 'x_test.npy'))
x_test_images = np.load(os.path.join(volume_path, 'x_test_images.npy'))
y_test = np.load(os.path.join(volume_path, 'y_test.npy'))
y_test_images = np.load(os.path.join(volume_path, 'y_test_images.npy'))

x_test_images = x_test_images[..., np.newaxis]


print(f'Test data - flat: {x_test_flat.shape}, images: {x_test_images.shape}')

min_accuracy = 0.90
min_f1 = 0.90
print(f'Quality gates: accuracy >= {min_accuracy}, F1 >= {min_f1}')

In [ ]:
# Get run IDs from previous tasks
try:
    linear_run_id = dbutils.jobs.taskValues.get(taskKey='train_linear_model', key='linear_run_id')
except:
    linear_run_id = None

try:
    xgb_run_id = dbutils.jobs.taskValues.get(taskKey='train_xgboost_model', key='xgboost_run_id')
except:
    xgb_run_id = None

try:
    nn_run_id = dbutils.jobs.taskValues.get(taskKey='train_neural_model', key='neural_run_id')
except:
    nn_run_id = None

print(f'Run IDs - Linear: {linear_run_id}, XGB: {xgb_run_id}, NN: {nn_run_id}')


In [ ]:
# Evaluate all models
import pandas as pd

results = []

# Logistic Regression
if linear_run_id:
    try:
        model = mlflow.sklearn.load_model(f'runs:/{linear_run_id}/model')
        y_pred = model.predict(x_test_flat)
        y_proba = model.predict_proba(x_test_flat)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        results.append({'model': 'logistic', 'run_id': linear_run_id, 'accuracy': acc, 'f1': f1, 'auc': auc})
        print(f'Logistic - Acc: {acc:.4f}, F1: {f1:.4f}')
    except Exception as e:
        logger.error(f'Logistic failed: {e}')

# XGBoost
if xgb_run_id:
    try:
        model = mlflow.xgboost.load_model(f'runs:/{xgb_run_id}/model')
        y_pred = model.predict(x_test_flat)
        y_proba = model.predict_proba(x_test_flat)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        results.append({'model': 'xgboost', 'run_id': xgb_run_id, 'accuracy': acc, 'f1': f1, 'auc': auc})
        print(f'XGBoost - Acc: {acc:.4f}, F1: {f1:.4f}')
    except Exception as e:
        logger.error(f'XGBoost failed: {e}')

# CNN
if nn_run_id:
    try:
        model = mlflow.tensorflow.load_model(f'runs:/{nn_run_id}/model')
        y_proba = model.predict(x_test_images, verbose=0)
        y_pred = np.argmax(y_proba, axis=1)
        acc = accuracy_score(y_test_images, y_pred)
        f1 = f1_score(y_test_images, y_pred, average='weighted')
        auc = roc_auc_score(y_test_images, y_proba, multi_class='ovr', average='weighted')
        results.append({'model': 'cnn', 'run_id': nn_run_id, 'accuracy': acc, 'f1': f1, 'auc': auc})
        print(f'CNN - Acc: {acc:.4f}, F1: {f1:.4f}')
    except Exception as e:
        logger.error(f'CNN failed: {e}')

# Show results and select best model
if results:
    df = pd.DataFrame(results)
    df['passes'] = (df['accuracy'] >= min_accuracy) & (df['f1'] >= min_f1)
    print('\n' + df.to_string(index=False))
    
    # Best model by F1 score
    best_idx = df['f1'].idxmax()
    best = df.loc[best_idx]
    print(f'Best: {best["model"].upper()} - F1: {best["f1"]:.4f}')
    
    # Set task values for downstream tasks (register_model)
    passes_threshold = bool(best['passes'])
    try:
        dbutils.jobs.taskValues.set(key='best_run_id', value=best['run_id'])
        dbutils.jobs.taskValues.set(key='best_algorithm', value=best['model'])
        dbutils.jobs.taskValues.set(key='best_f1', value=str(best['f1']))
        dbutils.jobs.taskValues.set(key='best_accuracy', value=str(best['accuracy']))
        dbutils.jobs.taskValues.set(key='passes_threshold', value=str(passes_threshold))
        print(f'Best model info passed to next task')
    except Exception as e:
        logger.warning(f'Could not set task values: {e}')